### Importación de Librerías y Definición de Variables

In [1]:
import requests
import pandas as pd
import numpy as np
import os

mapeo_latin_1 = {'Ã\x81':'Á', 'Ã\x89':'É', 'Ã\x8d':'Í', 'Ã\x93': 'Ó', 'Ã\x9a':'Ú', 'Ã\x91': "Ñ"}
mapeo_cambiar_tildes = {'Á':'A', 'É':'E', 'Í':'I', 'Ó':'O', 'Ú':'U'}
UMBRAL_TIPO_PROCEDIMIENTO = 100000


### 0. Extracción de data

In [2]:
# Creamos headers para evitar el bloqueo de la solicitud por parte del servidor
cabeceras = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

URL = "https://www.datosabiertos.gob.pe/api/3/action/package_show?id=30932ab9-2ee8-44d6-98c3-64628397ebc9"

resp = requests.get(URL, headers=cabeceras, timeout=10)

if resp.status_code == 200:
    resultado_api_formato_json = resp.json()
    lista_de_diccionarios_donde_estan_csv = resultado_api_formato_json["result"][0]["resources"]
else:
    print("Error al capturar los datos.")

df = pd.DataFrame(lista_de_diccionarios_donde_estan_csv)

lista_dataframes = []
lista_url = []

# VAMOS A DECLARAR LA SERIE CON LOS NOMBRES EJEMPLARES PARA NORMALIZAR LOS NOMBRES DE LAS COLUMNAS EN TODOS LOS DATASETS 
# Leemos este dataset que tiene todos los nombres bien escritos y extraemos los nombres de sus columnas
dataframe_202201 = pd.read_csv(
    'https://www.datosabiertos.gob.pe/sites/default/files/ReportePCBienes202201.csv', 
    sep=";", 
    index_col=False, 
    storage_options={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0'},
    encoding='latin-1'
    )

# Definimos los nombres de columnas a estandarizar cada dataset en cada csv
LISTA_NOMBRE_COLUMNAS_EJEMPLAR = dataframe_202201.columns.to_list() 

for url, duplicado in zip(df["url"], df["url"].duplicated()):
    if duplicado == True: # Si un url lo marca como duplicado, lo ignora, evitando la conversión del csv a dataframe
        pass
    elif ".pdf" in url: # Ignoramos el diccionario de datos 
        pass
    else:
        df = pd.read_csv(url, storage_options={'User-Agent': 'Mozilla/5.0'}, encoding='latin-1', sep=';')
        df.columns = LISTA_NOMBRE_COLUMNAS_EJEMPLAR
        lista_dataframes.append(df)
        lista_url.append(url)

# Unimos todos los dataframes generados por el bucle y almacenados en la lista 'lista_dataframes'
df = pd.concat(lista_dataframes, ignore_index=True) # ---> Parámetro = True porque sino nuestros index estarán duplicados 

# Enviamos la data cruda a un archivo csv para su almacenamiento
df.to_csv("Ordenes_Compra.csv", sep=";", mode='w', index=False)

### 1. Lectura del csv

In [3]:
df = pd.read_csv("Ordenes_Compra.csv", sep=";", low_memory=False, encoding='utf-8')
df.head(5)

,FECHA_PROCESO,RUC_PROVEEDOR,PROVEEDOR,RUC_ENTIDAD,ENTIDAD,TIPO_PROCEDIMIENTO,ORDEN_ELECTRÓNICA,ORDEN_ELECTRÓNICA_GENERADA,ESTADO_ORDEN_ELECTRÓNICA,DOCUMENTO_ESTADO_OCAM,FECHA_FORMALIZACIÓN,FECHA_ÚLTIMO_ESTADO,SUB_TOTAL,IGV,TOTAL,ORDEN_DIGITALIZADA,DESCRIPCIÓN_ESTADO,DESCRIPCIÓN_CESIÓN_DERECHOS,ACUERDO_MARCO
0,2022-05-12 12:51:18,20530182038,GRUPO COMPUDATA PERUSOCIEDAD DE RESPONSABILIDA...,20172606777,UNIVERSIDAD NACIONAL DE PIURA,Compra ordinaria,OCAM-2021-99-55-0,https://apps1.perucompras.gob.pe//OrdenCompra/...,PAGADA,https://saeusceprod01.blob.core.windows.net/co...,2022-01-04 00:00:00,2022-02-18 11:11:28,9298.90,1673.80,10972.70,https://saeusceprod01.blob.core.windows.net/co...,NaN,NaN,"IM-CE-2020-5 COMPUTADORAS DE ESCRITORIO, COMPU..."
1,2022-05-12 12:51:18,20530182038,GRUPO COMPUDATA PERUSOCIEDAD DE RESPONSABILIDA...,20172606777,UNIVERSIDAD NACIONAL DE PIURA,Compra ordinaria,OCAM-2021-99-57-0,https://apps1.perucompras.gob.pe//OrdenCompra/...,PAGADA,https://saeusceprod01.blob.core.windows.net/co...,2022-01-04 00:00:00,2022-02-18 11:11:05,2894.79,521.06,3415.85,https://saeusceprod01.blob.core.windows.net/co...,NaN,NaN,"EXT-CE-2021-6 IMPRESORAS, CONSUMIBLES, REPUEST..."
2,2022-05-12 12:51:18,20427497888,COMERCIAL DENIA S.A.C.,20174950971,UNIVERSIDAD NACIONAL DE EDUCACION ENRIQUE GUZM...,Compra ordinaria,OCAM-2021-106-327-0,https://apps1.perucompras.gob.pe//OrdenCompra/...,PAGADA,https://saeusceprod01.blob.core.windows.net/co...,2022-01-04 00:00:00,2022-03-30 11:20:38,22640.80,4075.34,26716.14,https://saeusceprod01.blob.core.windows.net/co...,NaN,NaN,"EXT-CE-2021-6 IMPRESORAS, CONSUMIBLES, REPUEST..."
3,2022-05-12 12:51:18,20600257286,PROYECTEC E.I.R.L.,20174950971,UNIVERSIDAD NACIONAL DE EDUCACION ENRIQUE GUZM...,Compra ordinaria,OCAM-2021-106-328-0,https://apps1.perucompras.gob.pe//OrdenCompra/...,ACEPTADA,NaN,2022-01-04 00:00:00,2022-01-04 00:07:35,11220.78,2019.74,13240.52,https://saeusceprod01.blob.core.windows.net/co...,NaN,NaN,"IM-CE-2020-5 COMPUTADORAS DE ESCRITORIO, COMPU..."
4,2022-05-12 12:51:18,20600668791,PAPELERIA REAL OFFICE E.I.R.L.,20174950971,UNIVERSIDAD NACIONAL DE EDUCACION ENRIQUE GUZM...,Compra ordinaria,OCAM-2021-106-340-0,https://apps1.perucompras.gob.pe//OrdenCompra/...,ACEPTADA,NaN,2022-01-04 00:00:00,2022-01-04 00:07:35,4586.57,825.58,5412.15,https://saeusceprod01.blob.core.windows.net/co...,NaN,NaN,"EXT-CE-2021-7 ÚTILES DE ESCRITORIO, PAPELES Y ..."


### 2. Info elemental del Dataframe

In [4]:
# Tamaño
display(df.shape)
# Resumen de la naturaleza de los datos de cada columna
display(df.info())
# Porcentaje de nulos
display(df.isnull().mean())
# Resumen de columnas strings
display(df.describe(include=["str"]))
# Resumen de columnas numéricas
pd.set_option('display.float_format', lambda x: '%.2f' % x)
display(df.describe(include=["int64", "float64"]))

(474845, 19)

<class 'pandas.DataFrame'>
RangeIndex: 474845 entries, 0 to 474844
Data columns (total 19 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   FECHA_PROCESO                474845 non-null  str    
 1   RUC_PROVEEDOR                474845 non-null  int64  
 2   PROVEEDOR                    474845 non-null  str    
 3   RUC_ENTIDAD                  474845 non-null  int64  
 4   ENTIDAD                      474845 non-null  str    
 5   TIPO_PROCEDIMIENTO           474845 non-null  str    
 6   ORDEN_ELECTRÓNICA            474845 non-null  str    
 7   ORDEN_ELECTRÓNICA_GENERADA   474845 non-null  str    
 8   ESTADO_ORDEN_ELECTRÓNICA     474845 non-null  str    
 9   DOCUMENTO_ESTADO_OCAM        12944 non-null   str    
 10  FECHA_FORMALIZACIÓN          474845 non-null  str    
 11  FECHA_ÚLTIMO_ESTADO          474845 non-null  str    
 12  SUB_TOTAL                    474845 non-null  float64
 13  IGV       

None

FECHA_PROCESO                  0.000000
RUC_PROVEEDOR                  0.000000
PROVEEDOR                      0.000000
RUC_ENTIDAD                    0.000000
ENTIDAD                        0.000000
TIPO_PROCEDIMIENTO             0.000000
ORDEN_ELECTRÓNICA              0.000000
ORDEN_ELECTRÓNICA_GENERADA     0.000000
ESTADO_ORDEN_ELECTRÓNICA       0.000000
DOCUMENTO_ESTADO_OCAM          0.972741
FECHA_FORMALIZACIÓN            0.000000
FECHA_ÚLTIMO_ESTADO            0.000000
SUB_TOTAL                      0.000000
IGV                            0.000000
TOTAL                          0.000000
ORDEN_DIGITALIZADA             0.000000
DESCRIPCIÓN_ESTADO             0.792452
DESCRIPCIÓN_CESIÓN_DERECHOS    0.978663
ACUERDO_MARCO                  0.000000
dtype: float64

,FECHA_PROCESO,PROVEEDOR,ENTIDAD,TIPO_PROCEDIMIENTO,ORDEN_ELECTRÓNICA,ORDEN_ELECTRÓNICA_GENERADA,ESTADO_ORDEN_ELECTRÓNICA,DOCUMENTO_ESTADO_OCAM,FECHA_FORMALIZACIÓN,FECHA_ÚLTIMO_ESTADO,ORDEN_DIGITALIZADA,DESCRIPCIÓN_ESTADO,DESCRIPCIÓN_CESIÓN_DERECHOS,ACUERDO_MARCO
count,474845,474845,474845,474845,474845,474845,474845,12944,474845,474845,474845,98553,10132,474845
unique,116,4868,4265,4,474845,474845,7,2822,74281,118793,474845,6,1,95
top,2022-05-12 13:02:27,TRADING SERVICE M&A SRLTDA,MARINA DE GUERRA DEL PERU,Compra ordinaria,OCAM-2021-99-55-0,https://apps1.perucompras.gob.pe//OrdenCompra/...,ACEPTADA,\t,2023-06-16 00:00:00,2023-06-16 00:20:28,https://saeusceprod01.blob.core.windows.net/co...,Actualizado automáticamente por la vinculación...,\t,"EXT-CE-2021-7 ÚTILES DE ESCRITORIO, PAPELES Y ..."
freq,16746,3574,8235,432269,1,1,373612,10123,1244,1060,1,88409,10132,82823


,RUC_PROVEEDOR,RUC_ENTIDAD,SUB_TOTAL,IGV,TOTAL
count,474845.00,474845.00,474845.00,474845.00,474845.00
mean,18987776474.29,20300699854.55,14311.71,2487.59,16799.30
std,3657494111.61,172075878.38,156399.01,28061.18,184377.75
min,10000223234.00,20100003199.00,0.04,0.00,0.05
25%,20455474184.00,20154469941.00,909.00,142.13,1065.54
50%,20600997352.00,20193065938.00,2295.54,378.00,2690.40
75%,20606376678.00,20487911586.00,6960.00,1170.00,8142.00
max,20613386875.00,20614761459.00,37779917.52,6800385.15,44580302.67


### 3. Limpieza de data

1° Eliminación de columnas que no influyen en el análisis, debido a los siguientes motivos:
- ESTADO_ORDEN_ELECTRÓNICA: La columna presenta 5 opciones de estado de las órdenes, siento 'ACEPTADA' el valor predominante en 380,000 registros. Dado que este estado precede al pago, una prevalencia de tal magnitud es inconsistente con la operación normal del gasto público, por lo que se concluye que los registros están desactualizados en origen y se excluye del modelo de datos
- FECHA_ÚLTIMO_ESTADO: Esta columna se enncuentra directamente relacionada con 'ESTADO_ORDEN_ELECTRÓNICA' y como prescindiremos de ella, entonces tampoco esta columna representa alguna utilidad.
- FECHA_PROCESO: Debido a que solo contiene la información de cuándo fueron cargados los datos al sistema,  más no tiene nada relacionado a la operación de las órdenes.  
- ORDEN_ELECTRÓNICA_GENERADA: Cumple la misma función que la columna 'ORDEN_DIGITALIZADA' de servir de sustento físico y legal a los datos registrados en el dataset, solo que lo muestra en otro formato. Por eso a efectos de rendimiento y simplicidad solo elegimos trabajar con la columna anteriormente mencionada.
- DOCUMENTO_ESTADO_OCAM: Destinada a registrar el comprobante de pago, presenta cobertura en solo 13,000 de los 92,000 registros marcados como 'PAGADA', representando una tasa de llenado del 14%. Su utilidad para validar el estado real de las órdenes es prácticamente nula
- DESCRIPCIÓN_ESTADO: Contiene casi un 80% de registros nulos, y los pocos que sí los tienen contienen un mensaje automatizado de la plataforma PERU_COMPRAS: Actualizado automáticamente por la vinculación con el SIAF. No presenta ninguna influencia en el análisis.
- DESCRIPCIÓN_CESIÓN_DERECHOS: Presenta casi un 98% de registros nulos, y el 2% restante solo contiene el registro de comando tabulador '\t'. 


In [5]:
df = df.drop(columns=["ESTADO_ORDEN_ELECTRÓNICA", "FECHA_ÚLTIMO_ESTADO", "FECHA_PROCESO", "ORDEN_ELECTRÓNICA_GENERADA", "DOCUMENTO_ESTADO_OCAM", 
                      "DESCRIPCIÓN_ESTADO", "DESCRIPCIÓN_CESIÓN_DERECHOS"])

2° Conversión las columnas a su tipo de dato correcto

In [6]:
# Las columnas que contienen el RUC de la empresa pese a ser identificadores, están como dato numérico, así que se cambian a dato string
df["RUC_PROVEEDOR"] = df["RUC_PROVEEDOR"].astype("str")
df["RUC_ENTIDAD"] = df["RUC_ENTIDAD"].astype("str")

In [7]:
# Las columnas que contienen fechas se cambian a tipo datetime, porque originalmente estaban como tipo texto
columnas_fecha = [condicion for condicion in df.columns.to_list() if "FECHA" in condicion and "FORMATEADA" not in condicion]

for columna_fecha in columnas_fecha:
    df[f"{columna_fecha}_FORMATEADA"] = pd.to_datetime(df[columna_fecha].str.strip().str.split(" ").str[0], format='mixed')
    print(f"La columna original era tipo {df[columna_fecha].dtype}, ahora la nueva es {df[f"{columna_fecha}_FORMATEADA"].dtype}")

La columna original era tipo str, ahora la nueva es datetime64[us]


3° Estandarización de columnas

A) Estandarización para columnas strings  
Primero realizamos la estandarización base

In [8]:
lista_columnas_strings = df.select_dtypes("string").columns.to_list()
for columna in lista_columnas_strings:
        # Las fechas venian como dato string pero ya hicimos los cambios correspondientes por eso las ignoramos aquí
    if "FECHA" in columna: 
        pass
    elif columna == "ORDEN_DIGITALIZADA":
        # La columna contiene links urls que dan acceso directo a la factura de la orden, por eso no ponemos todo en mayúscula aquí
        df[f"{columna}_LIMPIO"] = df[columna].str.strip().str.replace(mapeo_latin_1) 
    else:
        df[f"{columna}_LIMPIO"] = df[columna].str.strip().str.upper().str.replace(mapeo_latin_1)

Resumen de columnas strings originales y estandarizadas:  

In [9]:
df.describe(include="str")

,RUC_PROVEEDOR,PROVEEDOR,RUC_ENTIDAD,ENTIDAD,TIPO_PROCEDIMIENTO,ORDEN_ELECTRÓNICA,FECHA_FORMALIZACIÓN,ORDEN_DIGITALIZADA,ACUERDO_MARCO,RUC_PROVEEDOR_LIMPIO,PROVEEDOR_LIMPIO,RUC_ENTIDAD_LIMPIO,ENTIDAD_LIMPIO,TIPO_PROCEDIMIENTO_LIMPIO,ORDEN_ELECTRÓNICA_LIMPIO,ORDEN_DIGITALIZADA_LIMPIO,ACUERDO_MARCO_LIMPIO
count,474845,474845,474845,474845,474845,474845,474845,474845,474845,474845,474845,474845,474845,474845,474845,474845,474845
unique,3563,4868,2333,4265,4,474845,74281,474845,95,3563,3589,2333,3249,2,474845,474845,72
top,20107903951,TRADING SERVICE M&A SRLTDA,20153408191,MARINA DE GUERRA DEL PERU,Compra ordinaria,OCAM-2021-99-55-0,2023-06-16 00:00:00,https://saeusceprod01.blob.core.windows.net/co...,"EXT-CE-2021-7 ÚTILES DE ESCRITORIO, PAPELES Y ...",20107903951,TRADING SERVICE M&A SRLTDA,20153408191,MARINA DE GUERRA DEL PERU,COMPRA ORDINARIA,OCAM-2021-99-55-0,https://saeusceprod01.blob.core.windows.net/co...,"EXT-CE-2021-7 ÚTILES DE ESCRITORIO, PAPELES Y ..."
freq,3661,3574,8419,8235,432269,1,1244,1,82823,3661,3661,8419,8419,441267,1,1,86052


Verificamos si existen registros de RUCs que difieran de la ley (que tengan mayor o menor de 11 dígitos):

In [10]:
display(df["RUC_ENTIDAD_LIMPIO"].map(lambda fila: len(fila)!=11).sum())
display(df["RUC_PROVEEDOR_LIMPIO"].map(lambda fila: len(fila)!=11).sum())

np.int64(0)

np.int64(0)

Limpieza con reglas exclusivas a ACUERDO_MARCO_LIMPIO, ya que se encontraron duplicados debido a espacios extra intermedios, registros iguales pero con la diferencia que unos tenían un código adjunto y otros no, y unos contenían tildes y otros no.

In [11]:
# Aquí Usamos el diccionario para mapear y cambiar, Eliminamos espacios intermedios e Identificamos la estructura del código del acuerdo para quitarla
df["ACUERDO_MARCO_LIMPIO"] = df["ACUERDO_MARCO_LIMPIO"].str.replace(r'\w{2,3}-\w{2}-\d{4}-\d{1,2} ', '', regex=True).str.replace(r'\s+', ' ', regex=True).str.replace(mapeo_cambiar_tildes)

Limpieza con reglas exclusivas a TIPO_PROCEDIMIENTO_LIMPIO (se identificó que ofrece una segmentación de la data en base a reglas de negocio oficiales)
Se encontraron registros que rompían lógica de negocio de la plataforma PERU COMPRAS:
Donde el tipo 'COMPRA ORDINARIA' corresponde a montos totales menores a S/. 100 000, y el tipo 'GRAN COMPRA' corresponde a montos totales mayores o iguales.

In [12]:
df["TIPO_PROCEDIMIENTO_LIMPIO"] = np.where(df["TOTAL"]<UMBRAL_TIPO_PROCEDIMIENTO, "COMPRA ORDINARIA", "GRAN COMPRA")
Registros_incorrectos = len(df.loc[((df["TIPO_PROCEDIMIENTO_LIMPIO"]=="COMPRA ORDINARIA") & (df["TOTAL"]>=UMBRAL_TIPO_PROCEDIMIENTO)) | ((df["TIPO_PROCEDIMIENTO_LIMPIO"]=="GRAN COMPRA") & (df["TOTAL"]<UMBRAL_TIPO_PROCEDIMIENTO))])
print(f"Comprobación: La cantidad de filas que rompen la regla después de limpiar ahora son: {Registros_incorrectos}")

Comprobación: La cantidad de filas que rompen la regla después de limpiar ahora son: 0


B) Estandarización para columnas numéricas

Se encontraron registros que incumplen la relacion entre la suma de subtotal + igv = total.  
La diferencia entre la correcta suma de las columnas 'subtotal' e 'igv' comparada con la columna 'total', es tan despreciable e insignificante que no implica el crear una columna calculada.  
Se evidencia que la diferencia es producida a la manera inherente que tienen las computadoras de expresar los decimales en el sistema binario

In [13]:
# Validamos lógica de negocio: el subtotal + el igv de una orden serán igual al monto total
print(f"La cantidad de registros que incumplen la lógica de negocio son {(df["SUB_TOTAL"]+df["IGV"] != df["TOTAL"]).sum()}")
print(f"El monto máximo de diferencia que existe es: {(df["SUB_TOTAL"]+df["IGV"]-df["TOTAL"]).max()}")
print(f"El monto mínimo de diferencia que existe es: {(df["SUB_TOTAL"]+df["IGV"]-df["TOTAL"]).min()}")

La cantidad de registros que incumplen la lógica de negocio son 76775
El monto máximo de diferencia que existe es: 1.862645149230957e-09
El monto mínimo de diferencia que existe es: -3.725290298461914e-09


Se descubrió que, pese a abarcar una baja cantidad del total de registros (aproximadamente el 2%), el tipo de procedimiento de 'GRAN COMPRA' representa casi el 55% de la suma total de dinero de todas las órdenes.

In [14]:
print(f"La suma de monto TOTAL de las órdenes catalogadas como GRAN COMPRA:\nEs de S/.{df.loc[df["TOTAL"]>=UMBRAL_TIPO_PROCEDIMIENTO]["TOTAL"].sum()}\nRepresenta el {(df.loc[df["TOTAL"]>=UMBRAL_TIPO_PROCEDIMIENTO]["TOTAL"].sum()/df["TOTAL"].sum())*100}% de la suma total\nAbarca {(len(df.loc[df["TIPO_PROCEDIMIENTO_LIMPIO"] == "GRAN COMPRA"])/len(df))*100}% del total de registros")
print(f"La suma de monto TOTAL de las órdenes catalogadas como COMPRA ORDINARIA:\nEs de S/.{df.loc[df["TOTAL"]<UMBRAL_TIPO_PROCEDIMIENTO]["TOTAL"].sum()}\nRepresentan el {(df.loc[df["TOTAL"]<UMBRAL_TIPO_PROCEDIMIENTO]["TOTAL"].sum()/df["TOTAL"].sum())*100}% de la suma total\nAbarca {(len(df.loc[df["TIPO_PROCEDIMIENTO_LIMPIO"] == "COMPRA ORDINARIA"])/len(df))*100}% del total de registros")

La suma de monto TOTAL de las órdenes catalogadas como GRAN COMPRA:
Es de S/.4337621809.71
Representa el 54.376165237209605% de la suma total
Abarca 2.2013499141825226% del total de registros
La suma de monto TOTAL de las órdenes catalogadas como COMPRA ORDINARIA:
Es de S/.3639442756.7
Representan el 45.62383476279038% de la suma total
Abarca 97.79865008581747% del total de registros


Tipo fechas

Verificación de registros que rompan la lógica de tiempo, no se encontró ningún registro que incumpla la lógica

In [15]:
# Si alguno registra un mes que tenga un día que sobre pase el límite permitido (incluyendo los febreros bisiestos) con dt.days_in_month
fechas_dia_invalido  = df.loc[(df["FECHA_FORMALIZACIÓN_FORMATEADA"].dt.day > df["FECHA_FORMALIZACIÓN_FORMATEADA"].dt.days_in_month) |  
                              (df["FECHA_FORMALIZACIÓN_FORMATEADA"].dt.day < 1)]

# Si alguno registra un mes que tenga un mes fuera de los normales 
fechas_mes_invalido = df.loc[(df["FECHA_FORMALIZACIÓN_FORMATEADA"].dt.month > 12) | (df["FECHA_FORMALIZACIÓN_FORMATEADA"].dt.month < 1)]

# Si algún registro supera la fecha actual
fechas_futuras = df.loc[df["FECHA_FORMALIZACIÓN_FORMATEADA"] > pd.Timestamp.today().normalize()]

print(f"La cantidad de registros de fechas con un día inválido son: {len(fechas_dia_invalido)}")
print(f"La cantidad de registros de fechas con un mes inválido (mayor a (12) diciembre o menor a (1) enero) son: {len(fechas_mes_invalido)}")
print(f"La cantidad de registros de fechas sobrepasen la fecha actual son: {len(fechas_futuras)}")

La cantidad de registros de fechas con un día inválido son: 0
La cantidad de registros de fechas con un mes inválido (mayor a (12) diciembre o menor a (1) enero) son: 0
La cantidad de registros de fechas sobrepasen la fecha actual son: 0


## Validación de cambios:

7. En relación a los outliers:
   Se descubrió que existen tipos de precio EXTREMADAMENTE BAJOS para la naturaleza de la plataforma PERU COMPRAS, esto se debe a 2 motivos que se verificaron:  
        a) Relación precio - producto irrazonable: se encontraron registros de productos en las facturas que tenían un precio absurdamente disminuido para su naturaleza, por ejemplo, el registro del monto TOTAL más bajo registrado en el dataset (de S/.0.040) le pertenece a una laptop, este tipo de producto normalmente no baja de S/.1000  
        b) Relación precio - producto razonable: contrario a lo que se piensa, la plataforma PERU COMPRAS, pese a:  
            - Tener montos mínimos de compra según el tipo de acuerdo marco  
            - Servir de intermediaria para entidades del estado (donde las compras y los negocios adoptan montos altos), permite explícitamente la compra de productos menores al permitido  
           Permite la compra directa entre PROVEEDOR y ENTIDAD si es que los montos de la órden son menores de los permitidos por la plataforma, lo que justifica precios de montos pequeños (desde S/.1 hasta S/.500)

In [16]:
# Establecemos cálculos:
cols_antiguas_y_nuevas = {"PROVEEDOR": "PROVEEDOR_LIMPIO", "ENTIDAD": "ENTIDAD_LIMPIO", "TIPO_PROCEDIMIENTO": "TIPO_PROCEDIMIENTO_LIMPIO",
                          "ORDEN_ELECTRÓNICA":"ORDEN_ELECTRÓNICA_LIMPIO", "ACUERDO_MARCO": "ACUERDO_MARCO_LIMPIO"}

i = 0
for columna in cols_antiguas_y_nuevas:
    i += 1
    metrica = (df[columna] != df[cols_antiguas_y_nuevas[columna]]).sum()
    mensaje = f"{i}) Para Columna {columna}: {metrica} filas.\n\tRepresentaba un {((metrica*100)/len(df)):.2f}% del total de registros."
                  
    if (columna == "ORDEN_ELECTRÓNICA"):
        print(mensaje)
    else:
        print(mensaje)
        print(f"\tDe {df[columna].nunique()} registros únicos a solo {df[cols_antiguas_y_nuevas[columna]].nunique()}.")


1) Para Columna PROVEEDOR: 12437 filas.
	Representaba un 2.62% del total de registros.
	De 4868 registros únicos a solo 3589.
2) Para Columna ENTIDAD: 13491 filas.
	Representaba un 2.84% del total de registros.
	De 4265 registros únicos a solo 3249.
3) Para Columna TIPO_PROCEDIMIENTO: 474845 filas.
	Representaba un 100.00% del total de registros.
	De 4 registros únicos a solo 2.
4) Para Columna ORDEN_ELECTRÓNICA: 10132 filas.
	Representaba un 2.13% del total de registros.
5) Para Columna ACUERDO_MARCO: 391114 filas.
	Representaba un 82.37% del total de registros.
	De 95 registros únicos a solo 30.


## Creación de un dataframe con la data ya procesada


In [17]:
# Le pasamos al nuevo DataFrame solo las columnas limpias
columnas_eliminar = ["RUC_PROVEEDOR", "PROVEEDOR", "RUC_ENTIDAD", "ENTIDAD", "TIPO_PROCEDIMIENTO", "ORDEN_ELECTRÓNICA", 
                     "ORDEN_DIGITALIZADA", "ACUERDO_MARCO", "FECHA_FORMALIZACIÓN", "ORDEN_DIGITALIZADA_LIMPIO"]
df_limpio = df.drop(columns=columnas_eliminar)
df_limpio.info()

<class 'pandas.DataFrame'>
RangeIndex: 474845 entries, 0 to 474844
Data columns (total 11 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   SUB_TOTAL                       474845 non-null  float64       
 1   IGV                             474845 non-null  float64       
 2   TOTAL                           474845 non-null  float64       
 3   FECHA_FORMALIZACIÓN_FORMATEADA  474845 non-null  datetime64[us]
 4   RUC_PROVEEDOR_LIMPIO            474845 non-null  str           
 5   PROVEEDOR_LIMPIO                474845 non-null  str           
 6   RUC_ENTIDAD_LIMPIO              474845 non-null  str           
 7   ENTIDAD_LIMPIO                  474845 non-null  str           
 8   TIPO_PROCEDIMIENTO_LIMPIO       474845 non-null  str           
 9   ORDEN_ELECTRÓNICA_LIMPIO        474845 non-null  str           
 10  ACUERDO_MARCO_LIMPIO            474845 non-null  str           
dty

Verificamos la existencia de duplicados en el df_limpio (ya que es el que contiene toda la data estandarizada y uniforme en sus columnas para poder efectuar comparaciones a nivel de filas)

In [18]:
lista_df_sin_PK = ["FECHA_FORMALIZACIÓN_FORMATEADA", "SUB_TOTAL", "IGV", "TOTAL", "RUC_PROVEEDOR_LIMPIO", "PROVEEDOR_LIMPIO", "RUC_ENTIDAD_LIMPIO", "ENTIDAD_LIMPIO", 
                   "TIPO_PROCEDIMIENTO_LIMPIO", "ACUERDO_MARCO_LIMPIO"]

print(f"La cantidad de duplicados a nivel de dataframe entero son: {df_limpio.duplicated().sum()}")
print(f"Órdenes distintas con atributos transaccionales coincidentes: {df_limpio.duplicated(subset=lista_df_sin_PK).sum()}")

La cantidad de duplicados a nivel de dataframe entero son: 0
Órdenes distintas con atributos transaccionales coincidentes: 6483


Verificamos cardinalidades entre los RUCs y sus razones sociales
Descubrimientos:
1) Las entidades nacionales y los proveedores por igual actualizaron a lo largo del tiempo el nombre de su razon social, las entidades nacionales en mayor medida que las empresas proveedoras
2) Las entidades nacionales presentan casos donde el mismo nombre de razón social está asociado a distintos RUCs. Esto podría interpretarse como una violación a la unicidad de razón social, sin embargo se validó que corresponde a entidades homónimas ubicadas en distintas jurisdicciones geográficas del territorio peruano, lo cual es legalmente válido dado que las entidades públicas se identifican por su RUC y no por su denominación.

Con esto se concluye que el RUC constituye el identificador único e inequívoco de cada entidad, independientemente de su denominación, siendo el criterio correcto para construir las dimensiones del modelo de datos.

In [19]:
pares_columnas = {"RUC_PROVEEDOR_LIMPIO": "PROVEEDOR_LIMPIO", "RUC_ENTIDAD_LIMPIO": "ENTIDAD_LIMPIO"}

for ruc_col, nombre_col in pares_columnas.items():
    cardinalidad = df_limpio.groupby(ruc_col)[nombre_col].nunique()
    cardinalidad_inversa = df_limpio.groupby(nombre_col)[ruc_col].nunique()
    
    ocurrencias = len(cardinalidad.loc[cardinalidad > 1])
    ocurrencias_inversa = len(cardinalidad_inversa.loc[cardinalidad_inversa > 1])
    
    print(f"{ruc_col}: máximo {cardinalidad.max()} nombres por RUC, ocurre en {ocurrencias} RUCs")
    print(f"{nombre_col}: máximo {cardinalidad_inversa.max()} RUCs por nombre, ocurre en {ocurrencias_inversa} nombres")
    print()

RUC_PROVEEDOR_LIMPIO: máximo 2 nombres por RUC, ocurre en 26 RUCs
PROVEEDOR_LIMPIO: máximo 1 RUCs por nombre, ocurre en 0 nombres

RUC_ENTIDAD_LIMPIO: máximo 3 nombres por RUC, ocurre en 940 RUCs
ENTIDAD_LIMPIO: máximo 4 RUCs por nombre, ocurre en 28 nombres



# Normalizamos la data 

##  Creamos las tablas de dimensiones

In [20]:
orden_df = "FECHA_FORMALIZACIÓN_FORMATEADA"
# Para RUC_PROVEEDOR y PROVEEDOR
dim_proveedor = df_limpio.sort_values(by = [orden_df]).drop_duplicates(subset="RUC_PROVEEDOR_LIMPIO", keep='last')[["RUC_PROVEEDOR_LIMPIO", "PROVEEDOR_LIMPIO"]].sort_values(by=["RUC_PROVEEDOR_LIMPIO"])
dim_proveedor["key_proveedor"] = range(1, len(dim_proveedor) + 1)
orden_cols_prov = ["key_proveedor", "RUC_PROVEEDOR_LIMPIO", "PROVEEDOR_LIMPIO"]
dim_proveedor = dim_proveedor[orden_cols_prov]
# Para RUC_ENTIDAD y ENTIDAD
dim_entidad = df_limpio.sort_values(by = [orden_df]).drop_duplicates(subset="RUC_ENTIDAD_LIMPIO", keep='last')[["RUC_ENTIDAD_LIMPIO", "ENTIDAD_LIMPIO"]].sort_values(by=["RUC_ENTIDAD_LIMPIO"])
dim_entidad["key_entidad"] = range(1, len(dim_entidad) + 1)
orden_cols_ent = ["key_entidad", "RUC_ENTIDAD_LIMPIO", "ENTIDAD_LIMPIO"]
dim_entidad = dim_entidad[orden_cols_ent]
# ACUERDO_MARCO_LIMPIO
dim_acuerdo_marco = df_limpio.sort_values(by = [orden_df]).drop_duplicates(subset="ACUERDO_MARCO_LIMPIO", keep ='last')[["ACUERDO_MARCO_LIMPIO"]].sort_values(by=["ACUERDO_MARCO_LIMPIO"])
dim_acuerdo_marco["key_acuerdo"] = range(1, len(dim_acuerdo_marco) + 1)
orden_cols_acuerdo = ["key_acuerdo", "ACUERDO_MARCO_LIMPIO"]
dim_acuerdo_marco = dim_acuerdo_marco[orden_cols_acuerdo]


## Generamos la tabla de hechos

In [21]:
Fact_Ordenes = df_limpio.merge(dim_proveedor, on='RUC_PROVEEDOR_LIMPIO').merge(dim_entidad, on='RUC_ENTIDAD_LIMPIO').merge(dim_acuerdo_marco, on='ACUERDO_MARCO_LIMPIO')
columnas_eliminar = [columna for columna in Fact_Ordenes.columns.to_list() if "_x" in columna or "_y" in columna or "RUC" in columna or columna == "ACUERDO_MARCO_LIMPIO"]
Fact_Ordenes = Fact_Ordenes.drop(columns=columnas_eliminar)
orden_columnas = ["ORDEN_ELECTRÓNICA_LIMPIO"] + [col for col in Fact_Ordenes.columns.to_list() if col != "ORDEN_ELECTRÓNICA_LIMPIO"]
Fact_Ordenes = Fact_Ordenes[orden_columnas]

# Exportamos las tablas de dimensiones y hechos a sus respectivos CSV que almacenaran esos datos


In [22]:
# Creamos la carpeta que almacenará los csv
os.makedirs("Data_Normalizada", exist_ok=True)
# Para Fact_Ordenes
Fact_Ordenes.to_csv(r"Data_Normalizada\Fact_Ordenes.csv", sep=";", index=False, mode='w')
# Para dim_proveedor
dim_proveedor.to_csv(r"Data_Normalizada\dim_proveedor.csv", sep=";", index=False, mode='w')
# Para dim_entidad
dim_entidad.to_csv(r"Data_Normalizada\dim_entidad.csv", sep=";", index=False, mode='w')
# Para dim_acuerdo_marco
dim_acuerdo_marco.to_csv(r"Data_Normalizada\dim_acuerdo_marco.csv", sep=";", index=False, mode='w')